<div dir="rtl">

# 🎯 03 - RAG Chains with Retrievers & LCEL

## ما هي سلاسل الـ RAG عبر LCEL؟
- **RAG (Retrieval-Augmented Generation)**: تقنية تزويد النموذج اللغوي بسياق خارجي موثق مأخوذ من مستودع متجهات للإجابة بدقة ومنع الهلوسة.
- في بنية LCEL الحديثة، نربط الـ Retriever داخل السلسلة ليعمل بالتوازي مع تمرير السؤال، ثم تجميع المستندات عبر دالة تنسيق بسيطة (`format_docs`) وتمريرها للقالب.

</div>

<div dir="rtl">

### ⚙️ تهيئة البيئة ونموذج التضمين والمستودع المتجهي
نقوم بتحميل البيانات التجريبية وبناء مستودع FAISS محلي.

</div>

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# تحميل البيئة
load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# مستندات معرفية تجريبية حول LangChain ومكوناتها
sample_docs = [
    Document(
        page_content="LangSmith is an end-to-end observability and evaluation platform by LangChain. It provides full tracing for prompts, LLM calls, latency, and token consumption.",
        metadata={"source": "langsmith_docs", "topic": "observability"}
    ),
    Document(
        page_content="LCEL (LangChain Expression Language) provides a unified declarative syntax using the pipe operator (|) for composing chains with native streaming and async support.",
        metadata={"source": "lcel_docs", "topic": "syntax"}
    ),
    Document(
        page_content="FAISS is an open-source library created by Meta for dense vector similarity search and clustering. It operates entirely in-memory with optional local serialization.",
        metadata={"source": "faiss_docs", "topic": "vector_stores"}
    )
]

# بناء مستودع FAISS وتحويله إلى Retriever
vectorstore = FAISS.from_documents(sample_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ تم بناء مستودع FAISS وتجهيز الـ Retriever بنجاح!")


<div dir="rtl">

### 1️⃣ دالة تنسيق المستندات (Format Docs Callable)
دالة مساعدة بسيطة تأخذ قائمة المستندات المسترجعة وتجمع نصوصها مع الاستشهاد بالمصادر.

</div>

In [ ]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"[Doc {i} - Source: {source}]:\n{doc.page_content}")
    return "\n\n".join(formatted)

# اختبار الدالة على استعلام سريع
test_docs = retriever.invoke("What is LangSmith?")
print(format_docs(test_docs))


<div dir="rtl">

### 2️⃣ صياغة قالب RAG موجه لمنع الهلوسة
نصمم قالباً يفرض على النموذج الإجابة حصرياً بالاعتماد على السياق المرفق والاعتراف بعدم المعرفة إذا لم يتوفر السياق.

</div>

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an expert AI technical assistant. Answer the user question based strictly on the provided Context.
If the context does not contain enough information to answer, state clearly: 'Information not available in context.'

Context:
{context}

Question: {question}

Helpful & Accurate Answer:"""
)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
print("✅ تم إعداد قالب الـ RAG والنموذج بنجاح!")


<div dir="rtl">

### 3️⃣ بناء وتنفيذ سلسلة RAG عبر LCEL
نقوم بربط الـ Retriever مع دالة التنسيق والقالب والنموذج في خط أنابيب متكامل.

</div>

In [ ]:
# بناء سلسلة LCEL RAG المتكاملة
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# اختبار سؤال موجود في السياق
query_1 = "What is LangSmith and what does it monitor?"
print(f"❓ السؤال الأول: {query_1}")
print("💬 إجابة السلسلة:")
print(rag_chain.invoke(query_1))

print("\n" + "="*60 + "\n")

# اختبار سؤال غير موجود في السياق لفحص الحماية من الهلوسة
query_2 = "Who won the 2022 FIFA World Cup?"
print(f"❓ السؤال الثاني: {query_2}")
print("💬 إجابة السلسلة:")
print(rag_chain.invoke(query_2))


<div dir="rtl">

## 💡 خلاصة وخاتمة (Summary & Next Steps)
- تعلمنا كيفية ربط الـ Retriever داخل سلسلة LCEL بدون فئات وسيطة معقدة.
- نسقنا السياق مع المصادر وفرضنا حماية دقيقة ضد الهلوسة.
- **الخطوة التالية**: الانتقال إلى كراس `04_end_to_end_web_rag_app.ipynb` لبناء تطبيق GenAI كامل يستوعب صفحات حقيقية من الإنترنت ويجيب عنها تفاعلياً.

</div>